# ML Algorithms Lab Practical — Decision Tree Classifier

**Classifier 2 in the series.** This notebook is based on the uploaded *Machine Learning Algorithms Cheatsheet*, including the exact Decision Tree hyperparameter ranges printed in the cheat sheet's **Parameter Grids** section (page 5).

## Hyperparameters from the cheat sheet — all included

- `max_depth`: `[None, 3, 5, 7, 10]`
- `min_samples_split`: `[2, 5, 10]`
- `min_samples_leaf`: `[1, 2, 4]`
- `criterion`: `["gini", "entropy"]`

The cheat sheet's Decision Tree syntax also names `random_state`; this is fixed at `42` for reproducibility rather than searched. `class_weight` is included as an explicitly labelled extra option to help with imbalanced classes; it is not listed in the cheat-sheet grid.

## Notebook coverage
- CSV loading, data inspection, missing values, duplicates, target distribution and EDA plots.
- Leakage-safe numeric/categorical preprocessing.
- Stratified train/test split.
- Mandatory `GridSearchCV` and stratified cross-validation.
- PCA vs no-PCA comparison using matched outer folds.
- Accuracy, balanced accuracy, precision, recall, F1, MCC, Cohen's kappa, log loss and ROC-AUC where supported.
- Confusion matrices, ROC/precision-recall curves, learning curve, tree feature importance, grid diagnostics.
- Paired t-test and Wilcoxon signed-rank test on matched fold scores.

**Important:** The cheat sheet does not prescribe a Decision Tree grid for `class_weight`, PCA comparison, or exact statistical test workflow; these are supplementary practical additions. Tree models do not require scaling, and PCA may reduce interpretability or performance, but the comparison is included for consistent practical reporting. The notebook is syntax-checked, not run on your actual dataset.

In [ ]:
# ============================================================
# 0. IMPORTS AND REPRODUCIBILITY
# ============================================================
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel, wilcoxon
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
    matthews_corrcoef, cohen_kappa_score, log_loss, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve, auc
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme()
print("Imports complete. Random state:", RANDOM_STATE)


# ============================================================
# 1. LOAD DATA AND INITIAL INSPECTION
# ============================================================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Found one CSV:", default_path)
    use_default = input("Use this file? [Y/n]: ").strip().lower()
    file_path = default_path if use_default in ("", "y", "yes") else input("Enter CSV path: ").strip()
elif len(csv_files) > 1:
    print("CSV files in current folder:")
    for i, f in enumerate(csv_files, 1):
        print(f"{i}. {f}")
    file_path = input("Enter CSV filename/path: ").strip()
else:
    file_path = input("Enter CSV file path: ").strip()

df = pd.read_csv(file_path)
print("Shape:", df.shape)
display(df.head())
print("\nInfo:")
df.info()
print("\nDescriptive summary:")
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nColumn names:", df.columns.tolist())


# ============================================================
# 2. TARGET SELECTION, CLEANING, EDA
# ============================================================
print("Available columns:")
for col in df.columns:
    print(" -", col)
TARGET = input("\nEnter the exact classification target column: ").strip()
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' is not in the dataframe.")

df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[df[TARGET].notna()].copy()
y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

# Remove likely identifier columns to avoid learning row IDs.
id_like = []
for col in X.columns:
    name = str(col).lower()
    unique_ratio = X[col].nunique(dropna=True) / max(len(X), 1)
    if name in {"id", "record_id", "row_id", "customer_id", "employee_id", "sample_id"}:
        id_like.append(col)
    elif name.endswith("_id") and unique_ratio > 0.80:
        id_like.append(col)
    elif unique_ratio > 0.98 and X[col].nunique(dropna=True) > 20:
        id_like.append(col)
if id_like:
    print("Dropping likely ID columns:", id_like)
    X = X.drop(columns=id_like)

if y.nunique() < 2:
    raise ValueError("Classification requires at least two classes.")
print("\nClass counts:")
display(y.value_counts().to_frame("count"))
print("\nClass proportions:")
display(y.value_counts(normalize=True).to_frame("proportion"))

plt.figure(figsize=(8, 4))
sns.countplot(x=y.astype(str), order=y.astype(str).value_counts().index)
plt.title("Target Class Distribution")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

num_eda = X.select_dtypes(include=np.number).columns.tolist()
cat_eda = X.select_dtypes(exclude=np.number).columns.tolist()
if num_eda:
    X[num_eda].hist(figsize=(14, 9), bins=25)
    plt.suptitle("Numeric Feature Histograms")
    plt.tight_layout()
    plt.show()
    if len(num_eda) > 1:
        plt.figure(figsize=(10, 7))
        sns.heatmap(X[num_eda].corr(), cmap="coolwarm", center=0)
        plt.title("Numeric Feature Correlation")
        plt.tight_layout()
        plt.show()
for col in cat_eda[:8]:
    plt.figure(figsize=(8, 3))
    X[col].astype(str).value_counts().head(15).plot(kind="bar")
    plt.title(f"Top categories: {col}")
    plt.tight_layout()
    plt.show()


# ============================================================
# 3. STRATIFIED TRAIN/TEST SPLIT
# ============================================================
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError("A class has fewer than 2 samples; stratified train/test split is not reliable.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
display(y_train.value_counts(normalize=True).rename("train_proportion").to_frame())
display(y_test.value_counts(normalize=True).rename("test_proportion").to_frame())


# ============================================================
# 4. LEAKAGE-SAFE PREPROCESSING
# ============================================================
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_ohe())
])
preprocess_parts = []
if num_cols:
    preprocess_parts.append(("num", num_pipe, num_cols))
if cat_cols:
    preprocess_parts.append(("cat", cat_pipe, cat_cols))
if not preprocess_parts:
    raise ValueError("No usable features found.")
preprocessor = ColumnTransformer(preprocess_parts, remainder="drop")

# PCA comparison: tree model does not need scaling, but numeric values are scaled before PCA.
pca_parts = []
if len(num_cols) >= 2:
    pca_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE))
    ])
    pca_parts.append(("num", pca_num_pipe, num_cols))
elif num_cols:
    pca_parts.append(("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols))
if cat_cols:
    pca_parts.append(("cat", cat_pipe, cat_cols))
pca_preprocessor = ColumnTransformer(pca_parts, remainder="drop")
print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)


# ============================================================
# 5. DECISION TREE HYPERPARAMETER GRID — EXACT CHEAT-SHEET RANGES
# ============================================================
# Cheat sheet page 5:
# max_depth: [None, 3, 5, 7, 10]
# min_samples_split: [2, 5, 10]
# min_samples_leaf: [1, 2, 4]
# criterion: ["gini", "entropy"]
# random_state is also named in the model syntax; fixed at 42.
#
# Optional additional grid dimension (not specified in cheat-sheet grid):
# class_weight: None or "balanced".
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [None, 3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced"]
}
print("Cheat-sheet combinations (excluding optional class_weight):", 2 * 5 * 3 * 3)
print("Search combinations including optional class_weight:", 2 * 5 * 3 * 3 * 2)
print("Fixed random_state:", RANDOM_STATE)


# ============================================================
# 6. NESTED STRATIFIED CV: PCA VS NO PCA, SAME OUTER FOLDS
# ============================================================
min_train_class = int(y_train.value_counts().min())
outer_splits = min(5, min_train_class)
if outer_splits < 2:
    raise ValueError("Not enough samples per class for cross-validation.")

outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
fold_rows = []
fold_predictions = {}

for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    min_inner_class = int(y_tr.value_counts().min())
    inner_splits = min(3, min_inner_class)
    if inner_splits < 2:
        print(f"Skipping fold {fold}: too few examples per class for inner CV.")
        continue
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)

    for representation, prep in [("Without PCA", preprocessor), ("With PCA", pca_preprocessor)]:
        pipe = Pipeline([
            ("preprocess", clone(prep)),
            ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
        ])
        search = GridSearchCV(
            pipe, param_grid=param_grid, scoring="f1_macro",
            cv=inner_cv, n_jobs=-1, refit=True, return_train_score=True,
            error_score="raise"
        )
        t0 = time.time()
        search.fit(X_tr, y_tr)
        elapsed = time.time() - t0
        pred_tr = search.predict(X_tr)
        pred_va = search.predict(X_va)
        row = {
            "fold": fold,
            "representation": representation,
            "best_inner_cv_macro_f1": search.best_score_,
            "accuracy": accuracy_score(y_va, pred_va),
            "balanced_accuracy": balanced_accuracy_score(y_va, pred_va),
            "precision_macro": precision_score(y_va, pred_va, average="macro", zero_division=0),
            "recall_macro": recall_score(y_va, pred_va, average="macro", zero_division=0),
            "f1_macro": f1_score(y_va, pred_va, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_va, pred_va, average="weighted", zero_division=0),
            "train_macro_f1": f1_score(y_tr, pred_tr, average="macro", zero_division=0),
            "fit_seconds": elapsed,
            "best_params": search.best_params_
        }
        fold_rows.append(row)
        fold_predictions[(fold, representation)] = {
            "y_true": y_va.copy(), "y_pred": pd.Series(pred_va, index=y_va.index),
            "model": search.best_estimator_, "X_valid": X_va.copy()
        }
        print(f"\nFold {fold} | {representation}")
        print("Best params:", search.best_params_)
        print(f"Inner CV macro F1: {search.best_score_:.4f}")
        print(f"Validation macro F1: {row['f1_macro']:.4f}; accuracy: {row['accuracy']:.4f}; runtime: {elapsed:.1f}s")

cv_results = pd.DataFrame(fold_rows)
display(cv_results)
if cv_results.empty:
    raise RuntimeError("No cross-validation folds completed. Check class counts.")
summary_metrics = [
    "accuracy", "balanced_accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_weighted", "best_inner_cv_macro_f1", "fit_seconds"
]
print("\nCV mean and standard deviation:")
display(cv_results.groupby("representation")[summary_metrics].agg(["mean", "std"]))


# ============================================================
# 7. SIGNIFICANCE TESTS — PAIRED T-TEST AND WILCOXON
# ============================================================
wide = cv_results.pivot(index="fold", columns="representation", values="f1_macro").dropna()
if len(wide) >= 2 and {"Without PCA", "With PCA"}.issubset(wide.columns):
    scores_no = wide["Without PCA"].to_numpy()
    scores_pca = wide["With PCA"].to_numpy()
    t_stat, t_p = ttest_rel(scores_no, scores_pca)
    try:
        w_stat, w_p = wilcoxon(scores_no, scores_pca, zero_method="wilcox")
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    print("Paired PCA vs no-PCA comparison on matched outer-fold macro F1")
    print(f"Paired t-test: statistic={t_stat:.4f}, p={t_p:.6f}")
    print(f"Wilcoxon signed-rank: statistic={w_stat}, p={w_p}")
    print("At alpha=0.05:", "significant difference detected" if t_p < 0.05 else "no statistically significant difference detected")
    print("Caution: a small number of folds gives low statistical power; interpret tests cautiously.")
else:
    print("Not enough paired fold scores for significance tests.")


# ============================================================
# 8. CROSS-VALIDATION VISUALIZATIONS
# ============================================================
plt.figure(figsize=(8, 4))
sns.boxplot(data=cv_results, x="representation", y="f1_macro")
sns.stripplot(data=cv_results, x="representation", y="f1_macro", color="black", alpha=0.6)
plt.title("Outer-fold Macro F1: PCA vs No PCA")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.lineplot(data=cv_results, x="fold", y="accuracy", hue="representation", marker="o")
plt.title("Outer-fold Accuracy by Fold")
plt.tight_layout()
plt.show()

for representation in ["Without PCA", "With PCA"]:
    folds = [f for f in range(1, outer_splits + 1) if (f, representation) in fold_predictions]
    if folds:
        item = fold_predictions[(folds[-1], representation)]
        labels = np.unique(y_train)
        cm = confusion_matrix(item["y_true"], item["y_pred"], labels=labels)
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
            xticks_rotation=45, cmap="Blues", values_format="d"
        )
        plt.title(f"Fold {folds[-1]} Confusion Matrix — {representation}")
        plt.tight_layout()
        plt.show()


# ============================================================
# 9. FINAL MODEL: CHOOSE REPRESENTATION ON CV, TUNE TRAIN SET, TEST ONCE
# ============================================================
mean_f1 = cv_results.groupby("representation")["f1_macro"].mean().sort_values(ascending=False)
selected_representation = mean_f1.index[0]
selected_preprocessor = preprocessor if selected_representation == "Without PCA" else pca_preprocessor
print("Selected representation:", selected_representation)

final_splits = min(5, int(y_train.value_counts().min()))
final_cv = StratifiedKFold(n_splits=final_splits, shuffle=True, random_state=RANDOM_STATE)
final_pipe = Pipeline([
    ("preprocess", clone(selected_preprocessor)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])
final_search = GridSearchCV(
    final_pipe, param_grid=param_grid, scoring="f1_macro",
    cv=final_cv, n_jobs=-1, refit=True, return_train_score=True, error_score="raise"
)
t0 = time.time()
final_search.fit(X_train, y_train)
final_seconds = time.time() - t0
best_model = final_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nFINAL DECISION TREE MODEL")
print("Best parameters:", final_search.best_params_)
print(f"Best training CV macro F1: {final_search.best_score_:.4f}")
print(f"Final fit/tuning time: {final_seconds:.2f}s")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


# ============================================================
# 10. COMPREHENSIVE HOLD-OUT METRICS
# ============================================================
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "matthews_corrcoef": matthews_corrcoef(y_test, y_pred),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred)
}
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    metrics["log_loss"] = log_loss(y_test, proba, labels=best_model.classes_)
    try:
        if len(best_model.classes_) == 2:
            metrics["roc_auc"] = roc_auc_score(y_test, proba[:, 1])
            metrics["average_precision"] = average_precision_score(
                (y_test == best_model.classes_[1]).astype(int), proba[:, 1]
            )
        else:
            metrics["roc_auc_ovr_macro"] = roc_auc_score(
                y_test, proba, multi_class="ovr", average="macro", labels=best_model.classes_
            )
    except ValueError as exc:
        print("ROC-AUC/AP unavailable for this test split:", exc)

print("\nHeld-out test metrics:")
display(pd.DataFrame({"metric": list(metrics.keys()), "value": list(metrics.values())}).sort_values("metric"))


# ============================================================
# 11. TEST CONFUSION MATRIX + ROC / PRECISION-RECALL CURVES
# ============================================================
labels = best_model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    xticks_rotation=45, cmap="Blues", values_format="d"
)
plt.title("Final Test Confusion Matrix")
plt.tight_layout()
plt.show()

if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    if len(labels) == 2:
        positive_label = labels[1]
        y_binary = (y_test == positive_label).astype(int)
        fpr, tpr, _ = roc_curve(y_binary, proba[:, 1])
        roc_auc_value = auc(fpr, tpr)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc_value:.3f}")
        plt.plot([0, 1], [0, 1], "--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve (positive class={positive_label})")
        plt.legend()
        plt.tight_layout()
        plt.show()

        prec, rec, _ = precision_recall_curve(y_binary, proba[:, 1])
        ap = average_precision_score(y_binary, proba[:, 1])
        plt.figure(figsize=(6, 5))
        plt.plot(rec, prec, label=f"Average Precision={ap:.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        y_bin = label_binarize(y_test, classes=labels)
        plt.figure(figsize=(7, 5))
        for i, cls in enumerate(labels):
            if i < proba.shape[1] and 0 < y_bin[:, i].sum() < len(y_bin):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
                plt.plot(fpr, tpr, label=f"{cls} AUC={auc(fpr, tpr):.3f}")
        plt.plot([0, 1], [0, 1], "--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Multiclass ROC Curves (One-vs-Rest)")
        plt.legend()
        plt.tight_layout()
        plt.show()


# ============================================================
# 12. LEARNING CURVE + GRID SEARCH DIAGNOSTICS
# ============================================================
try:
    lc_cv = StratifiedKFold(n_splits=min(5, int(y_train.value_counts().min())), shuffle=True, random_state=RANDOM_STATE)
    train_sizes, train_scores, val_scores = learning_curve(
        clone(best_model), X_train, y_train, cv=lc_cv, scoring="f1_macro",
        train_sizes=np.linspace(0.3, 1.0, 5), n_jobs=-1,
        shuffle=True, random_state=RANDOM_STATE
    )
    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training macro F1")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation macro F1")
    plt.xlabel("Training examples")
    plt.ylabel("Macro F1")
    plt.title("Learning Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Learning curve skipped:", exc)

results_df = pd.DataFrame(final_search.cv_results_).sort_values("rank_test_score")
print("\nTop 10 parameter combinations:")
display(results_df[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10))
print("\nBest parameters:", final_search.best_params_)
print("Best CV macro F1:", final_search.best_score_)


# ============================================================
# 13. DECISION TREE FEATURE IMPORTANCE + TREE PLOT
# ============================================================
try:
    fitted_prep = best_model.named_steps["preprocess"]
    fitted_tree = best_model.named_steps["model"]
    feature_names = fitted_prep.get_feature_names_out()
    importances = pd.Series(fitted_tree.feature_importances_, index=feature_names).sort_values(ascending=False)
    print("Top feature importances:")
    display(importances.head(20).to_frame("importance"))
    plt.figure(figsize=(9, 6))
    importances.head(20).sort_values().plot(kind="barh")
    plt.title("Top Decision Tree Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    # Draw a readable truncated tree (max_depth limits only the display, not the fitted model).
    plt.figure(figsize=(18, 9))
    plot_tree(
        fitted_tree, max_depth=3, feature_names=feature_names,
        class_names=[str(c) for c in fitted_tree.classes_],
        filled=True, rounded=True, fontsize=7
    )
    plt.title("Final Decision Tree (display truncated to depth 3)")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Feature importance/tree visualization skipped:", exc)


# ============================================================
# 14. SIGNIFICANCE TEST NOTES
# ============================================================
print("The paired significance tests compare PCA vs no-PCA fold-level macro F1 on the same outer folds.")
print("For comparing different classifiers, run them on identical folds and compare the paired fold scores.")
print("Use p < 0.05 as a conventional threshold, not as proof that one model is universally better.")
